In [1]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles available:")
print(os.listdir("."))


StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 3, Finished, Available, Finished, False)

Current directory:
/mnt/var/hadoop/tmp/nm-secondary-local-dir/usercache/trusted-service-user/appcache/application_1789582864001_0001/container_1789582864001_0001_01_000001

Files available:
['container_tokens', '.default_container_executor_session.sh.crc', '.launch_container.sh.crc', 'launch_container.sh', '.container_tokens.crc', 'builtin', 'sparkr', 'tmp', '__spark_conf__', 'default_container_executor_session.sh', '.default_container_executor.sh.crc', 'default_container_executor.sh']


In [2]:
df = spark.read.text("Files/Bronze/jobs/powerbi_jobs_clean.csv")

df.show(10, truncate=False)

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 4, Finished, Available, Finished, False)

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                                                                     |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [3]:
jobs_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv("Files/Bronze/jobs/powerbi_jobs_clean.csv")
)

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 5, Finished, Available, Finished, False)

In [4]:
print("Rows:", jobs_df.count())

jobs_df.printSchema()

jobs_df.select(
    "job_id",
    "job_title",
    "company",
    "city",
    "country"
).show(10, truncate=False)

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 6, Finished, Available, Finished, False)

Rows: 20
root
 |-- job_id: string (nullable = true)
 |-- job_title: string (nullable = true)
 |-- description: string (nullable = true)
 |-- company: string (nullable = true)
 |-- publication_date: string (nullable = true)
 |-- application_deadline: string (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- working_hours: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- workplace_model: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- country: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- occupation_field: string (nullable = true)
 |-- job_url: string (nullable = true)

+--------+-----------------------------------------------------------+---------------------------+---------+-------+
|job_id  |job_title                                                  |company                    |city     |country|
+--------+-----------------------------------------------------------

In [5]:
bronze_jobs_df = (
    spark.read
    .option("header", True)
    .option("multiLine", True)
    .option("quote", '"')
    .option("escape", '"')
    .option("mode", "PERMISSIVE")
    .csv("Files/Bronze/jobs/powerbi_jobs_clean.csv")
)

print("Bronze rows:", bronze_jobs_df.count())

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 7, Finished, Available, Finished, False)

Bronze rows: 20


In [6]:
from pyspark.sql.functions import col, count, when, isnan

bronze_jobs_df.select(
    [
        count(
            when(
                col(c).isNull() | (col(c) == ""),
                c
            )
        ).alias(c)
        for c in bronze_jobs_df.columns
    ]
).show()

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 8, Finished, Available, Finished, False)

+------+---------+-----------+-------+----------------+--------------------+---------------+-------------+--------+---------------+----+------+-------+----------+----------------+-------+
|job_id|job_title|description|company|publication_date|application_deadline|employment_type|working_hours|duration|workplace_model|city|region|country|occupation|occupation_field|job_url|
+------+---------+-----------+-------+----------------+--------------------+---------------+-------------+--------+---------------+----+------+-------+----------+----------------+-------+
|     0|        0|          0|      0|               0|                   0|              0|            3|       3|              0|   0|     0|      0|         0|               0|      0|
+------+---------+-----------+-------+----------------+--------------------+---------------+-------------+--------+---------------+----+------+-------+----------+----------------+-------+



In [7]:
from pyspark.sql.functions import col

print("Total rows:", bronze_jobs_df.count())

print(
    "Unique job IDs:",
    bronze_jobs_df.select("job_id").distinct().count()
)

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 9, Finished, Available, Finished, False)

Total rows: 20
Unique job IDs: 20


In [8]:
bronze_jobs_df.groupBy("job_id") \
    .count() \
    .filter(col("count") > 1) \
    .show()

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 10, Finished, Available, Finished, False)

+------+-----+
|job_id|count|
+------+-----+
+------+-----+



In [9]:
bronze_jobs_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_jobs")

StatementMeta(, 92e26e19-17a9-43d1-8d7c-3d03129be91e, 11, Finished, Available, Finished, False)